# Knowledge Graph Ingestion (Schema.org Aligned)

This notebook ingests Microsoft shareholder letters into a Neo4j knowledge graph using a **schema.org-aligned schema** with financial extensions.

## Schema Overview

**Node Types:**
- `Organization` - Companies (Microsoft, OpenAI)
- `Person` - Executives (Satya Nadella)
- `Product` - Products/Services (Azure, Copilot)
- `MonetaryAmount` - Financial metrics ($211B revenue)
- `Strategy` - Strategic initiatives
- `Risk` - Business risks
- `Technology` - Technologies mentioned
- `Metric` - Non-monetary metrics (users, growth %)
- `Event` - Announcements, launches

**Why Schema.org?**
- Industry standard for structured data
- LLMs trained on schema.org terminology
- Financial types from FIBO extension (MonetaryAmount)
- Better than custom schema for interoperability

## 1. Setup & Imports

In [1]:
# Pydantic V1/V2 compatibility patch
import sys

try:
    from pydantic import v1 as pydantic_v1
except ImportError:
    import pydantic as pydantic_v1

sys.modules["langchain_core.pydantic_v1"] = pydantic_v1
print("✅ Pydantic patch applied")

✅ Pydantic patch applied


In [2]:
import os
import time
from dotenv import load_dotenv
from tqdm import tqdm

import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_neo4j import Neo4jGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

print("✅ All imports successful")

✅ All imports successful


In [55]:
# Load environment variables
load_dotenv()
print("✅ Environment loaded")

✅ Environment loaded


## 2. Connect to Neo4j AuraDB

In [4]:
print("🔌 Connecting to Neo4j...")

try:
    graph = Neo4jGraph(
        url=os.getenv("NEO4J_URI"),
        username=os.getenv("NEO4J_USERNAME"),
        password=os.getenv("NEO4J_PASSWORD")
    )
    print("✅ Connected to Neo4j successfully.")
except Exception as e:
    print(f"❌ Connection Failed: {e}")

🔌 Connecting to Neo4j...
✅ Connected to Neo4j successfully.


## 3. Clear Existing Graph (Optional but Recommended)

⚠️ **WARNING**: This will delete ALL existing data in the graph!

In [5]:
# Check what's currently in the graph
result = graph.query("MATCH (n) RETURN count(n) as node_count")
node_count = result[0]['node_count'] if result else 0

result = graph.query("MATCH ()-[r]->() RETURN count(r) as rel_count")
rel_count = result[0]['rel_count'] if result else 0

print(f"📊 Current graph has {node_count} nodes and {rel_count} relationships")

📊 Current graph has 600 nodes and 609 relationships


In [6]:
# Uncomment to clear the graph
CLEAR_GRAPH = True  # Set to True to clear existing data

if CLEAR_GRAPH and node_count > 0:
    print("🗑️ Clearing existing graph...")
    
    # Delete in batches to avoid timeout on large graphs
    batch_size = 10000
    deleted_total = 0
    
    while True:
        result = graph.query(f"""
            MATCH (n)
            WITH n LIMIT {batch_size}
            DETACH DELETE n
            RETURN count(*) as deleted
        """)
        deleted = result[0]['deleted'] if result else 0
        deleted_total += deleted
        
        if deleted == 0:
            break
        print(f"   Deleted {deleted_total} nodes...")
    
    print(f"✅ Cleared {deleted_total} nodes from graph")
else:
    print("⏭️ Skipping graph clear (CLEAR_GRAPH=False or graph empty)")

🗑️ Clearing existing graph...
   Deleted 600 nodes...
✅ Cleared 600 nodes from graph


## 4. Create Schema Constraints in Neo4j

In [7]:
# Create constraints for unique node IDs
constraints = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Organization) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Person) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Product) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:MonetaryAmount) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Strategy) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Risk) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Technology) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Metric) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Event) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Initiative) REQUIRE n.id IS UNIQUE",
]

# Create indexes for common queries
indexes = [
    "CREATE INDEX IF NOT EXISTS FOR (n:Organization) ON (n.name)",
    "CREATE INDEX IF NOT EXISTS FOR (n:Person) ON (n.name)",
    "CREATE INDEX IF NOT EXISTS FOR (n:Product) ON (n.name)",
    "CREATE INDEX IF NOT EXISTS FOR (n:MonetaryAmount) ON (n.period)",
]

print("📐 Creating schema constraints...")
for cypher in constraints + indexes:
    try:
        graph.query(cypher)
    except Exception as e:
        print(f"   Warning: {e}")

print("✅ Schema constraints created")

📐 Creating schema constraints...
✅ Schema constraints created


## 5. Initialize LLM (Gemini)

In [48]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
    google_api_key=os.getenv("GOOGLE_API_KEY"),  # ✅ Pass API key directly
    convert_system_message_to_human=True
)

print("✅ Gemini LLM initialized")

✅ Gemini LLM initialized


In [74]:
# ollama for local since free Gemini keep hitting limits:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    #model="qwen2.5:14b",  
    model="llama3.1:8b",
    temperature=0,
)

print("✅ Ollama LLM initialized")

✅ Ollama LLM initialized


In [67]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-70b-versatile",
    temperature=0,
)

print("✅ Groq LLM initialized (Llama 3.3 70B)")

✅ Groq LLM initialized (Llama 3.3 70B)


## 6. Load and Chunk Documents

In [68]:
# Load shareholder letters from ./data folder
print("📂 Loading shareholder letters from ./data...")

loader = DirectoryLoader(
    './data',
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)

try:
    raw_docs = loader.load()
    print(f"✅ Loaded {len(raw_docs)} documents")
    
    # Show file names
    for doc in raw_docs:
        filename = doc.metadata.get('source', 'unknown').split('/')[-1].split('\\')[-1]
        print(f"   - {filename} ({len(doc.page_content)} chars)")
except Exception as e:
    print(f"❌ Error loading files: {e}")

📂 Loading shareholder letters from ./data...
✅ Loaded 5 documents
   - 2020_shareholder_letter.txt (25199 chars)
   - 2021_shareholder_letter.txt (24896 chars)
   - 2022_shareholder_letter.txt (25692 chars)
   - 2023_shareholder_letter.txt (28867 chars)
   - 2024_shareholder_letter.txt (28551 chars)


In [69]:
# Split into chunks
print("✂️ Splitting text into chunks...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,  # Characters per chunk
    chunk_overlap=200  # Overlap for context continuity
)

documents = text_splitter.split_documents(raw_docs)
print(f"✅ Created {len(documents)} chunks for processing")

✂️ Splitting text into chunks...
✅ Created 80 chunks for processing


## 7. Configure Schema.org-Aligned Extractor

This is the key part - we define exactly which node types and relationships to extract, aligned with schema.org.

In [75]:
# Schema.org-aligned node types
ALLOWED_NODES = [
    # Core schema.org types
    "Organization",      # Companies, corporations
    "Person",            # Executives, founders
    "Product",           # Products, services
    "Event",             # Announcements, launches
    
    # Financial types (from FIBO schema.org extension)
    "MonetaryAmount",    # Revenue, investments, costs
    
    # Custom extensions for strategic analysis
    "Strategy",          # Strategic initiatives
    "Risk",              # Business risks
    "Technology",        # Technologies, platforms
    "Metric",            # Non-monetary metrics (users, growth %)
    "Initiative",        # Programs, campaigns

    # Adding to allow the LLM add these becuase of the error
    "Concept",        
    "Program",
]

# Relationship types (schema.org-aligned)
ALLOWED_RELATIONSHIPS = [
    # Organization relationships
    "CEO",               # Organization → Person
    "FOUNDER",           # Organization → Person  
    "PRODUCES",          # Organization → Product
    "ACQUIRED",          # Organization → Organization
    "PARTNERS_WITH",     # Organization → Organization
    "COMPETES_WITH",     # Organization → Organization
    
    # Financial relationships
    "HAS_REVENUE",       # Organization/Product → MonetaryAmount
    "HAS_INVESTMENT",    # Organization → MonetaryAmount
    "COSTS",             # Initiative → MonetaryAmount
    
    # Strategic relationships
    "FOCUSES_ON",        # Organization → Strategy/Technology
    "LAUNCHED",          # Organization → Product/Initiative
    "ANNOUNCED",         # Organization → Event
    "USES",              # Organization/Product → Technology
    
    # Risk relationships
    "FACES_RISK",        # Organization → Risk
    "MITIGATES",         # Strategy → Risk
    
    # Metric relationships
    "MEASURED_BY",       # Product/Strategy → Metric
    "INCREASED",         # Thing → Metric
    "DECREASED",         # Thing → Metric
    
    # Generic
    "RELATED_TO",        # Fallback
    "MENTIONS",          # Document → Thing

    # Adding some relations to bypass the errors
    "PROTECTS",       
    "SUPPORTS",       
    "INVESTS_IN"
]

print(f"📋 Schema defined:")
print(f"   - {len(ALLOWED_NODES)} node types")
print(f"   - {len(ALLOWED_RELATIONSHIPS)} relationship types")

📋 Schema defined:
   - 12 node types
   - 23 relationship types


In [76]:
# Custom extraction prompt optimized for schema.org
EXTRACTION_PROMPT = """
You are an expert knowledge graph builder extracting structured data from Microsoft shareholder letters.

Extract entities and relationships using this SCHEMA.ORG-ALIGNED schema:

## Node Types:
- **Organization**: Companies, corporations (Microsoft, OpenAI, Activision)
- **Person**: Executives, founders, key people (Satya Nadella, Brad Smith)
- **Product**: Products, services, platforms (Azure, Microsoft 365, Copilot, Xbox)
- **MonetaryAmount**: Financial values with currency ($211 billion revenue, $35B investment)
- **Strategy**: Strategic initiatives (AI-first strategy, Cloud expansion)
- **Risk**: Business risks and challenges (Cybersecurity threats, Regulatory compliance)
- **Technology**: Technologies, platforms (GPT-4, Machine Learning, Cloud Computing)
- **Metric**: Non-monetary metrics (300 million users, 29% growth, 32,000 nonprofits)
- **Event**: Announcements, launches, conferences (Build 2024, Ignite)
- **Initiative**: Programs, campaigns (AI for Good, Digital Skills)

## Relationship Types:
{rel_types}

## Guidelines:
1. Use the MOST SPECIFIC node type (Organization over generic)
2. For MonetaryAmount, extract the value, currency, and what it represents
3. For Metric, include the value, unit, and time period if mentioned
4. Create relationships ONLY when explicitly stated or strongly implied
5. Use consistent naming (e.g., "Microsoft" not "MSFT" or "microsoft")

## Examples:
- "Microsoft revenue was $211 billion" → Organization(Microsoft) -[HAS_REVENUE]→ MonetaryAmount($211B)
- "Satya Nadella, CEO of Microsoft" → Person(Satya Nadella) -[CEO]→ Organization(Microsoft)
- "Azure grew 29%" → Product(Azure) -[INCREASED]→ Metric(29% growth)
- "investing in AI" → Organization -[FOCUSES_ON]→ Technology(AI)

Node Labels: {node_labels}

Text to extract from:
{input}
"""

# Create prompt template
extraction_template = ChatPromptTemplate.from_messages([
    ("user", EXTRACTION_PROMPT),
])

# Pre-fill the schema values
prompt_with_schema = extraction_template.partial(
    node_labels=", ".join(ALLOWED_NODES),
    rel_types=", ".join(ALLOWED_RELATIONSHIPS)
)

print("✅ Extraction prompt configured")

✅ Extraction prompt configured


In [77]:
# Initialize the LLM Graph Transformer
llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=ALLOWED_NODES,
    allowed_relationships=ALLOWED_RELATIONSHIPS,
    prompt=prompt_with_schema,
    strict_mode=False,  # Allow some flexibility in extraction
)

print("✅ Graph transformer ready")

✅ Graph transformer ready


## 8. Extract Graph Data (with Rate Limiting)

This processes all chunks through the LLM to extract entities and relationships.

⏱️ **Note**: This will take ~10-15 minutes due to API rate limits.

In [78]:
print(f"🤖 Extracting graph data from {len(documents)} chunks...")
print("   (Processing with rate limiting to avoid quota errors)")

graph_documents = []
batch_size = 10  # Process 10 chunks at a time
delay_between_batches = 0  # Wait 60 seconds between batches (Gemini free tier)

# For tracking
total_nodes = 0
total_relationships = 0

try:
    for i in tqdm(range(0, len(documents), batch_size)):
        batch = documents[i:i+batch_size]
        
        # Process batch
        batch_results = llm_transformer.convert_to_graph_documents(batch)
        graph_documents.extend(batch_results)
        
        # Count extracted items
        for doc in batch_results:
            total_nodes += len(doc.nodes)
            total_relationships += len(doc.relationships)
        
        # Wait before next batch (except for the last batch)
        if i + batch_size < len(documents):
            print(f"\n   ⏳ Batch complete. Nodes: {total_nodes}, Rels: {total_relationships}")
            print(f"   ⏳ Waiting {delay_between_batches}s before next batch...")
            time.sleep(delay_between_batches)
    
    print(f"\n✅ Extraction Complete!")
    print(f"   - Processed {len(documents)} chunks")
    print(f"   - Extracted {total_nodes} nodes")
    print(f"   - Extracted {total_relationships} relationships")
    print(f"   - Created {len(graph_documents)} graph documents")

except Exception as e:
    print(f"❌ Extraction Error: {e}")
    print(f"   Processed {len(graph_documents)} documents before error")

🤖 Extracting graph data from 80 chunks...
   (Processing with rate limiting to avoid quota errors)


 12%|██████████▍                                                                        | 1/8 [03:25<23:55, 205.10s/it]


   ⏳ Batch complete. Nodes: 52, Rels: 43
   ⏳ Waiting 0s before next batch...


 25%|████████████████████▊                                                              | 2/8 [05:45<16:43, 167.22s/it]


   ⏳ Batch complete. Nodes: 90, Rels: 73
   ⏳ Waiting 0s before next batch...


...


   ⏳ Batch complete. Nodes: 283, Rels: 242
   ⏳ Waiting 0s before next batch...


 88%|████████████████████████████████████████████████████████████████████████▋          | 7/8 [21:23<03:07, 187.56s/it]


   ⏳ Batch complete. Nodes: 327, Rels: 280
   ⏳ Waiting 0s before next batch...


100%|███████████████████████████████████████████████████████████████████████████████████| 8/8 [24:00<00:00, 180.12s/it]


✅ Extraction Complete!
   - Processed 80 chunks
   - Extracted 363 nodes
   - Extracted 317 relationships
   - Created 80 graph documents


In [73]:
print(f"🤖 Extracting graph data from {len(documents)} chunks...")
print("   (Processing with rate limiting to avoid quota errors)")

graph_documents = []
batch_size = 10
delay_between_batches = 60  # Safe for Groq

total_nodes = 0
total_relationships = 0
failed_batches = 0

for i in tqdm(range(0, len(documents), batch_size)):
    batch = documents[i:i+batch_size]
    
    try:
        batch_results = llm_transformer.convert_to_graph_documents(batch)
        graph_documents.extend(batch_results)
        
        for doc in batch_results:
            total_nodes += len(doc.nodes)
            total_relationships += len(doc.relationships)
        
        print(f"\n   ✅ Batch complete. Nodes: {total_nodes}, Rels: {total_relationships}")
        
    except Exception as e:
        failed_batches += 1
        print(f"\n   ⚠️ Batch {i//batch_size + 1} failed: {str(e)[:100]}...")
        print(f"   Continuing with next batch...")
    
    # Wait before next batch
    if i + batch_size < len(documents):
        time.sleep(delay_between_batches)

print(f"\n✅ Extraction Complete!")
print(f"   - Processed {len(documents)} chunks")
print(f"   - Extracted {total_nodes} nodes")
print(f"   - Extracted {total_relationships} relationships")
print(f"   - Failed batches: {failed_batches}")

🤖 Extracting graph data from 80 chunks...
   (Processing with rate limiting to avoid quota errors)


  0%|                                                                                            | 0/8 [00:00<?, ?it/s]


   ⚠️ Batch 1 failed: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned...
   Continuing with next batch...


  0%|                                                                                            | 0/8 [01:00<?, ?it/s]


KeyboardInterrupt: 

## 9. Inspect Extracted Data

In [79]:
# Count nodes by type
from collections import Counter

node_types = Counter()
rel_types = Counter()

for doc in graph_documents:
    for node in doc.nodes:
        node_types[node.type] += 1
    for rel in doc.relationships:
        rel_types[rel.type] += 1

print("📊 Node Types Extracted:")
for node_type, count in node_types.most_common():
    print(f"   - {node_type}: {count}")

print("\n📊 Relationship Types Extracted:")
for rel_type, count in rel_types.most_common(15):
    print(f"   - {rel_type}: {count}")

📊 Node Types Extracted:
   - Organization: 137
   - Product: 101
   - Person: 72
   - Monetaryamount: 19
   - Technology: 14
   - Metric: 8
   - Initiative: 7
   - Event: 3
   - Risk: 1
   - Strategy: 1

📊 Relationship Types Extracted:
   - PRODUCES: 79
   - CEO: 72
   - USES: 38
   - PARTNERS_WITH: 28
   - FOCUSES_ON: 19
   - HAS_REVENUE: 16
   - HAS_INVESTMENT: 12
   - LAUNCHED: 9
   - ACQUIRED: 9
   - MENTIONS: 5
   - SUPPORTS: 4
   - INCREASED: 4
   - WORKS_FOR: 3
   - MEASURED_BY: 3
   - ANNOUNCED: 3


In [80]:
# Show sample nodes
print("📝 Sample Extracted Nodes:")

seen_types = set()
for doc in graph_documents[:20]:
    for node in doc.nodes:
        if node.type not in seen_types:
            seen_types.add(node.type)
            print(f"   [{node.type}] {node.id}")
            if node.properties:
                for k, v in list(node.properties.items())[:2]:
                    print(f"      {k}: {str(v)[:50]}")
        if len(seen_types) >= 10:
            break
    if len(seen_types) >= 10:
        break

📝 Sample Extracted Nodes:
   [Organization] Microsoft
   [Person] Satya Nadella
   [Product] Github
   [Technology] Gpt-3
   [Initiative] Ai For Health
   [Event] Covid-19
   [Monetaryamount] $1.9B
   [Metric] 243,000


## 10. Save to Neo4j

In [81]:
# Reconnect if needed
if 'graph' not in locals() or graph is None:
    print("🔌 Reconnecting to Neo4j...")
    graph = Neo4jGraph(
        url=os.getenv("NEO4J_URI"),
        username=os.getenv("NEO4J_USERNAME"),
        password=os.getenv("NEO4J_PASSWORD")
    )
    print("✅ Reconnected")

# Save to Neo4j
if graph_documents:
    print(f"💾 Saving {len(graph_documents)} documents to Neo4j...")
    
    try:
        graph.add_graph_documents(graph_documents)
        graph.refresh_schema()
        
        print("✅ Saved! Your Knowledge Graph is ready.")
        print("   → Go to your Neo4j Console (Explore tab) to visualize it.")
    except Exception as e:
        print(f"❌ Save Error: {e}")
else:
    print("⚠️ No documents to save. Did extraction complete successfully?")

💾 Saving 80 documents to Neo4j...
✅ Saved! Your Knowledge Graph is ready.
   → Go to your Neo4j Console (Explore tab) to visualize it.


## 11. Verify the Graph

In [82]:
# Verify what's in the graph
print("📊 Graph Statistics:")

# Total counts
result = graph.query("MATCH (n) RETURN count(n) as count")
print(f"   Total Nodes: {result[0]['count']}")

result = graph.query("MATCH ()-[r]->() RETURN count(r) as count")
print(f"   Total Relationships: {result[0]['count']}")

# Counts by node type
print("\n📊 Nodes by Type:")
result = graph.query("""
    MATCH (n)
    RETURN labels(n)[0] as type, count(*) as count
    ORDER BY count DESC
""")
for row in result:
    print(f"   - {row['type']}: {row['count']}")

📊 Graph Statistics:
   Total Nodes: 183
   Total Relationships: 198

📊 Nodes by Type:
   - Product: 62
   - Organization: 51
   - Monetaryamount: 20
   - Technology: 14
   - Metric: 11
   - Initiative: 8
   - Event: 6
   - Person: 3
   - Concept: 3
   - Strategy: 3
   - Country: 1
   - Risk: 1


In [83]:
# Sample query: Find Microsoft and its relationships
print("🔍 Sample Query: Microsoft's Relationships")

result = graph.query("""
    MATCH (m:Organization)-[r]->(n)
    WHERE toLower(m.id) CONTAINS 'microsoft'
    RETURN type(r) as relationship, labels(n)[0] as target_type, n.id as target
    LIMIT 15
""")

for row in result:
    print(f"   Microsoft --[{row['relationship']}]--> [{row['target_type']}] {row['target']}")

🔍 Sample Query: Microsoft's Relationships
   Microsoft --[FOCUSES_ON]--> [Product] Power Platform
   Microsoft --[FOCUSES_ON]--> [Initiative] Ai For Health
   Microsoft --[FOCUSES_ON]--> [Initiative] Ai For Accessibility
   Microsoft --[FOCUSES_ON]--> [Technology] Tech
   Microsoft --[FOCUSES_ON]--> [Initiative] Ai For Humanitarian Action
   Microsoft --[FOCUSES_ON]--> [Technology] Technology
   Microsoft --[FOCUSES_ON]--> [Concept] Digital Imperative
   Microsoft --[FOCUSES_ON]--> [Technology] Ai
   Microsoft --[FOCUSES_ON]--> [Strategy] Ai Platform Shift
   Microsoft --[FOCUSES_ON]--> [Metric] 80%
   Microsoft --[FOCUSES_ON]--> [Organization] Microsoft
   Microsoft --[MENTIONS]--> [Person] Satya Nadella
   Microsoft --[MENTIONS]--> [Concept] Gdpr
   Microsoft --[MENTIONS]--> [Event] Environmental Sustainability Report
   Microsoft --[MENTIONS]--> [Concept] Microsoft Employees


In [84]:
# Sample query: Find all MonetaryAmounts
print("💰 Sample Query: Financial Metrics")

result = graph.query("""
    MATCH (m:MonetaryAmount)
    RETURN m.id as metric, m.value as value, m.currency as currency, m.period as period
    LIMIT 10
""")

for row in result:
    print(f"   {row['metric']}: {row['value']} {row['currency'] or ''} ({row['period'] or 'N/A'})")

💰 Sample Query: Financial Metrics


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: value)} {position: line: 3, column: 30, offset: 59} for query: '\n    MATCH (m:MonetaryAmount)\n    RETURN m.id as metric, m.value as value, m.currency as currency, m.period as period\n    LIMIT 10\n'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is availa

In [85]:
# Sample query: Find Risks
print("⚠️ Sample Query: Risks Identified")

result = graph.query("""
    MATCH (r:Risk)
    OPTIONAL MATCH (org:Organization)-[:FACES_RISK]->(r)
    RETURN r.id as risk, collect(DISTINCT org.id) as faced_by
    LIMIT 10
""")

for row in result:
    orgs = ", ".join(row['faced_by']) if row['faced_by'] else "(general)"
    print(f"   {row['risk']} - faced by: {orgs}")

⚠️ Sample Query: Risks Identified
   Climate Change - faced by: Microsoft


## 12. Export Schema Info (Optional)

In [86]:
# Print the current schema
print("📋 Current Neo4j Schema:")
print(graph.schema)

📋 Current Neo4j Schema:
Node properties:
Technology {id: STRING}
Product {id: STRING}
Strategy {id: STRING}
Risk {id: STRING}
Person {id: STRING}
Concept {id: STRING}
Organization {id: STRING}
Metric {id: STRING}
Event {id: STRING}
Initiative {id: STRING}
Monetaryamount {id: STRING}
Country {id: STRING}
Relationship properties:

The relationships:
(:Product)-[:INCREASED]->(:Metric)
(:Product)-[:USES]->(:Technology)
(:Product)-[:USES]->(:Product)
(:Product)-[:HAS_REVENUE]->(:Organization)
(:Product)-[:HAS_REVENUE]->(:Monetaryamount)
(:Product)-[:PARTNERS_WITH]->(:Product)
(:Product)-[:FOCUSES_ON]->(:Organization)
(:Product)-[:MENTIONS]->(:Person)
(:Person)-[:CEO]->(:Organization)
(:Person)-[:WORKS_FOR]->(:Organization)
(:Person)-[:USES]->(:Product)
(:Organization)-[:FOCUSES_ON]->(:Product)
(:Organization)-[:FOCUSES_ON]->(:Initiative)
(:Organization)-[:FOCUSES_ON]->(:Technology)
(:Organization)-[:FOCUSES_ON]->(:Concept)
(:Organization)-[:FOCUSES_ON]->(:Strategy)
(:Organization)-[:FOCUSES

---

## ✅ Done!

Your knowledge graph has been created with the **schema.org-aligned schema**.

### Next Steps:
1. **Explore in Neo4j Browser** - Go to your AuraDB console and click "Explore"
2. **Run the Research Copilot** - The agent can now query this graph
3. **Community Detection** - Run `3_graph_analysis.ipynb` to find clusters

### Sample Cypher Queries to Try:

```cypher
// Find all products by Microsoft
MATCH (m:Organization {id: 'Microsoft'})-[:PRODUCES]->(p:Product)
RETURN p.id, p.description

// Find strategic focus areas
MATCH (m:Organization)-[:FOCUSES_ON]->(s:Strategy)
RETURN m.id, collect(s.id) as strategies

// Find revenue metrics
MATCH (org:Organization)-[:HAS_REVENUE]->(m:MonetaryAmount)
RETURN org.id, m.value, m.currency, m.period
ORDER BY m.value DESC
```